# Mizuseki, Sirota, Pastalkova & Buzsaki (2009) on CRCNS hc-2

Tests the claims of Mizuseki K, Sirota A, Pastalkova E, Buzsaki G (2009), "Theta Oscillations
Provide Temporal Windows for Local Circuit Computation in the Entorhinal-Hippocampal Loop",
Neuron 64, 267-280, doi:10.1016/j.neuron.2009.08.037, on the CRCNS data set hc-2
(doi:10.6080/K0Z60KZ9). The claims file `claims/mizuseki-2009.yaml` was written from the version
of record and its Supplemental Data (main text, legends, Supplemental Experimental Procedures,
Table S1).

**Data set and version.** CRCNS data sets are not versioned; hc-2 is recorded as `unversioned`
and the files read are pinned by their MD5 sums from the data set's `checksums.md5`, printed
below. This is the only published form of hc-2 (data descriptor version 1.2, 18 January 2012), so
there is no version to choose; the paper cites none. Each session's `<session>.tar.gz` (spike
times, clusters, waveforms, LFP, positions, .xml) is streamed from the public bucket; the
wideband `.dat` and the video are not read.

**What hc-2 holds.** 21 open-field sessions (180 or 120 cm square platform) of three rats (ec13,
ec14, ec16) with a 4- or 8-shank silicon probe in CA1, the CA1 part of the open-field data of the
paper. It holds no entorhinal, CA3 or dentate recordings, no linear track, T-maze or wheel
sessions, and no data of rats ec12 and i01. So the claims about EC layers, CA3, DG, other tasks
and phase precession on the linear track are blocked (`data_missing`), and the tested claims are
tested on CA1 alone. hc-2 has no EC3 LFP, so theta phase is referred to the CA1 pyramidal layer
LFP (the paper's reference for rat i01 and for Figure S10); CA1 theta leads EC3 theta by 0 to 80
degrees in the paper.

| Claim | Tested here |
|---|---|
| tables1-cell-counts | blocked: every expected count includes data hc-2 does not hold (counts shown) |
| fig3d-hippocampal-modulation | tested on CA1 (expected values) |
| fig3-interneurons-min-ascending | tested on CA1 |
| figs9-phase-method | tested on CA1 |
| fig5-rate-phase-shift | tested on CA1 |
| fig5b-burst-double-peak | tested on CA1 |
| figs13-interneuron-rate | tested on CA1 |
| figs12b-burst-single-phase | tested on CA1 |
| figs19-fast-firing-minority | tested on CA1 |
| figs7-individual-rats | tested on CA1, three rats |
| figs11-phase-time-agreement | tested on CA1 pairs |
| figs2-ripple-sequence | tested on CA1 (fold increases) |
| fig2c, figs4, fig1 (2), fig3 EC and CA3/DG claims (5), fig3d-ec2, figs10, fig4 (4), figs8, fig6, fig7a, figs15, fig7b, figs16, figs18, fig7, fig8, figs6, discussion | blocked: data missing (EC, CA3, DG, other tasks, Montgomery et al. CSD) |

Every tested outcome rests on these assumptions and is marked so in the record. Methods the paper
leaves open and the choices made (each an assumption, varied below where it
could change a result): the theta-epoch threshold (theta ratio above 2) and running speed (above
3 cm/s); the CA1 cell-type rule (trough-to-peak latency below 0.4 ms or mean rate above 10 Hz);
the pyramidal layer channel (largest ripple-band power among sound channels). Details are in the
docstring of `hc2.py`.

In [ ]:
import os, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt
import hc2

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("data set", hc2.DATASET, "doi", hc2.DOI)

## Loading

Each session's `<session>.tar.gz` is streamed once from the public bucket and reduced as it is
read (`hc2.load_session`); three sessions are streamed at a time. The MD5 sums of the files read
are printed from the data set's `checksums.md5`.

In [ ]:
md5 = hc2.checksums()
for s, rat, side, rew in hc2.SESSIONS:
    print(f"{s}/{s}.tar.gz  md5 {md5[f'{s}/{s}.tar.gz']}")

def _load(s):
    t = time.time()
    d = hc2.load_session(s)
    d["load_s"] = time.time() - t
    return d

with ProcessPoolExecutor(max_workers=3) as ex:
    RAW = {d["session"]: d for d in ex.map(_load, [s for s, *_ in hc2.SESSIONS])}
print(f"loaded {len(RAW)} sessions in {time.time() - T0:.0f} s; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
INFO = {s: (rat, side, rew) for s, rat, side, rew in hc2.SESSIONS}

def theta_all(ratio_thr=hc2.THETA_RATIO, speed_thr=hc2.SPEED_CM_S):
    """Theta reduction of every session; returns per-session results and a unit table."""
    TH, rows = {}, []
    for s, d in RAW.items():
        rat, side, rew = INFO[s]
        r = hc2.theta_session(d, side, ratio_thr, speed_thr)
        TH[s] = r
        for i, u in enumerate(r["units"]):
            mu, R, p = hc2.circ_stats(u["ph"])
            rows.append({"session": s, "rat": rat, "day": hc2.DAY[s], "shank": u["shank"],
                         "cluster": u["cluster"], "i": i, "n_spikes": u["n_spikes"],
                         "rate": u["rate"], "t2p": u["t2p"], "asym": u["asym"],
                         "burst_index": u["burst_index"], "n_theta": u["n_theta"],
                         "theta_rate": u["theta_rate"], "phase": mu, "R": R, "p": p})
    U = pd.DataFrame(rows)
    U["type"] = hc2.classify(U.t2p, U.rate)
    U["included"] = (U.n_theta >= 50) & (U.theta_rate > 0.1)
    U["modulated"] = U.included & (U.p < 0.01)
    return TH, U

TH, U = theta_all()
print(f"theta reduction done at {time.time() - T0:.0f} s")

## Data check

In [ ]:
print("Sessions, rats and duration (s) from the LFP, the .whl file and the last spike:")
rows = []
for s, d in RAW.items():
    rat, side, rew = INFO[s]
    last = max((u["spikes"][-1] for u in d["units"] if len(u["spikes"])), default=np.nan)
    unsorted = sum(int((np.diff(u["spikes"]) < 0).any()) for u in d["units"])
    rows.append({"session": s, "rat": rat, "day": hc2.DAY[s], "maze_cm": side, "reward": rew,
                 "n_channels": d["xml"]["n_channels"], "shanks": len(d["xml"]["spike_groups"]),
                 "lfp_s": d["eeg_samples"] / hc2.FS_LFP, "whl_s": len(d["whl"]) / hc2.FS_WHL,
                 "last_spike_s": last, "units": len(d["units"]),
                 "units_unsorted_times": unsorted,
                 "noise_spikes": d["n_noise_spikes"], "lfp_channel": d["lfp_channel"],
                 "theta_s": TH[s]["theta_s"], "theta_hz": TH[s]["theta_freq"],
                 "cm_per_px": TH[s]["scale_cm_px"], "load_s": d["load_s"]})
S = pd.DataFrame(rows)
print(S.round(2).to_string())
print("\nrats:", S.groupby("rat").session.count().to_dict(), " sessions:", len(S),
      " units (session by session):", int(S.units.sum()))
print("LFP and .whl durations agree within one video frame in every session:",
      bool((abs(S.lfp_s - S.whl_s) < 1 / hc2.FS_WHL + 1e-6).all()))
print("every spike falls within the LFP duration:", bool((S.last_spike_s <= S.lfp_s).all()))
print("recordings deposited twice? sessions sharing an LFP length:", int(S.duplicated("lfp_s").sum()),
      "; sharing a spike count:", int(pd.Series([sum(len(u["spikes"]) for u in d["units"]) for d in RAW.values()]).duplicated().sum()))
xml_meta = {s: {(u.get("structure", ""), u.get("type", "")) for u in d["xml"]["units_xml"]} for s, d in RAW.items()}
print("structure/type fields of the units in the .xml files:", set().union(*xml_meta.values()))
print("\nLFP reference channel (CA1 pyramidal layer, hc2._pick_pyramidal_channel): ripple-band RMS relative to the "
      "median channel, correlation with its shank, and channels left out as unsound:")
for s, d in RAW.items():
    cc = d["channel_check"]; c = d["lfp_channel"]
    print(f"  {s}: channel {c}, relative RMS {cc[c]['rel_rms']:.2f}, r with shank {cc[c]['r_shank']:.3f}; "
          f"left out {[k for k, v in cc.items() if not v['ok']]}")
print("\nmembers of each tarball (one session shown):", sorted(RAW["ec013.527"]["members"]))

### Units of the variables

Spike times are .res time stamps divided by 20 kHz (data descriptor), LFP is one .eeg channel at
1250 Hz in A/D units, positions are .whl pixels scaled by the platform side (`hc2.position_cm`).
The ranges below are what the analysis uses.

In [ ]:
allsp = np.concatenate([u["spikes"] for d in RAW.values() for u in d["units"]])
print(f"spike times: {allsp.min():.4f} to {allsp.max():.1f} s, {len(allsp)} spikes in isolated units")
lf = np.concatenate([d["lfp"][::50] for d in RAW.values()])
print(f"LFP (A/D units, every 50th sample): 1st-99th percentile {np.percentile(lf, 1):.0f} to {np.percentile(lf, 99):.0f}")
sp = np.concatenate([TH[s]["speed"] for s in TH])
print(f"speed (cm/s): median {np.nanmedian(sp):.1f}, 99th percentile {np.nanpercentile(sp, 99):.1f}, "
      f"tracking missing in {np.isnan(sp).mean() * 100:.1f}% of frames")
print(f"theta epochs: {sum(TH[s]['theta_s'] for s in TH) / 3600:.2f} h of {S.lfp_s.sum() / 3600:.2f} h; "
      f"theta frequency (median peak interval) {S.theta_hz.min():.2f} to {S.theta_hz.max():.2f} Hz")
print(f"pixel scale (cm per pixel): 180 cm platform {S[S.maze_cm == 180].cm_per_px.min():.3f} to "
      f"{S[S.maze_cm == 180].cm_per_px.max():.3f}; 120 cm platform {S[S.maze_cm == 120].cm_per_px.min():.3f} to "
      f"{S[S.maze_cm == 120].cm_per_px.max():.3f}")

### Do the streams share a clock?

Three contrasts. (1) Spikes and LFP: the fraction of included units significantly locked to theta
phase, with the LFP as recorded and with the LFP phase shifted by 10.3 s. (2) Position and LFP:
the correlation of running speed with the theta ratio, true and with the speed shifted by 60 s.
(3) Spikes and position: the spatial information of units in 10 cm bins, true and with positions
shifted by 60 s.

In [ ]:
def shifted_lock(s, shift_s=10.3):
    r, d = TH[s], RAW[s]
    y = hc2.theta_filter(d["lfp"])
    un = hc2.unwrap_phase(hc2.phase_extrema(y, "max")[0])
    n = int(shift_s * hc2.FS_LFP)
    un_sh = np.roll(un, n)
    out = []
    for u in r["units"]:
        if u["n_theta"] < 50:
            continue
        p_true = hc2.circ_stats(u["ph"])[2]
        p_sh = hc2.circ_stats(hc2.spike_phase(u["t"], un_sh))[2]
        out.append((p_true < 0.01, p_sh < 0.01))
    return np.array(out)

def speed_theta_corr(s, shift_s=0.0):
    t, ratio = hc2.theta_ratio(RAW[s]["lfp"])
    v = TH[s]["speed"]
    tw = np.arange(len(v)) / hc2.FS_WHL
    vi = np.interp(t + shift_s, tw, np.nan_to_num(v, nan=np.nan), left=np.nan, right=np.nan)
    ok = np.isfinite(vi)
    return stats.spearmanr(vi[ok], ratio[ok])[0]

def spatial_info(s, shift_s=0.0, bin_cm=10.0):
    d, r = RAW[s], TH[s]
    rat, side, rew = INFO[s]
    x, y, _ = hc2.position_cm(d["whl"], side)
    n = int(round(shift_s * hc2.FS_WHL))
    x, y = np.roll(x, n), np.roll(y, n)
    ok = np.isfinite(x) & np.isfinite(y)
    xb = np.floor((x - np.nanmin(x)) / bin_cm); yb = np.floor((y - np.nanmin(y)) / bin_cm)
    key = (xb * 1000 + yb)
    occ_keys, occ = np.unique(key[ok], return_counts=True)
    occ_t = occ / hc2.FS_WHL
    pocc = occ_t / occ_t.sum()
    res = []
    for u in d["units"]:
        fr = np.clip((u["spikes"] * hc2.FS_WHL).astype(int), 0, len(x) - 1)
        fr = fr[ok[fr]]
        if len(fr) < 100:
            continue
        ks, c = np.unique(key[fr], return_counts=True)
        cnt = np.zeros(len(occ_keys)); cnt[np.searchsorted(occ_keys, ks)] = c
        lam = cnt / occ_t; lm = (pocc * lam).sum()
        nz = lam > 0
        res.append((pocc[nz] * lam[nz] / lm * np.log2(lam[nz] / lm)).sum())
    return np.array(res)

rows = []
for s in TH:
    L = shifted_lock(s)
    si_t, si_s = spatial_info(s), spatial_info(s, 60.0)
    rows.append({"session": s, "units_50": len(L), "locked_true": L[:, 0].mean(), "locked_shifted": L[:, 1].mean(),
                 "speed_theta_rho_true": speed_theta_corr(s), "speed_theta_rho_shift60": speed_theta_corr(s, 60.0),
                 "spatial_info_true": np.median(si_t), "spatial_info_shift60": np.median(si_s)})
CLK = pd.DataFrame(rows)
print(CLK.round(3).to_string())
print("\nsessions where locking, speed-theta and spatial information all drop when shifted:",
      int(((CLK.locked_true > CLK.locked_shifted) & (CLK.speed_theta_rho_true > CLK.speed_theta_rho_shift60)
           & (CLK.spatial_info_true > CLK.spatial_info_shift60)).sum()), "of", len(CLK))

### Cell types

The paper separates hippocampal principal cells and interneurons by autocorrelograms, waveforms
and mean rates without giving thresholds; hc-2 has no cell-type labels (the `structure` and
`type` fields of the .xml files are empty). The main classification (`hc2.classify`) marks a unit
as a putative interneuron when its trough-to-peak latency is below 0.4 ms or its mean rate above
10 Hz (assumption). The table shows the counts and features under it and under two variants.

In [ ]:
print("trough-to-peak latency (ms) of all units:", np.round(np.sort(U.t2p.unique()), 2))
print(pd.crosstab(pd.cut(U.t2p, [0, 0.375, 0.425, 2], labels=["<0.4", "0.4", ">0.4"]),
                  pd.cut(U.rate, [0, 1, 5, 10, 1000]), margins=True))
print("\nfeatures by type (median):")
print(U.groupby("type")[["t2p", "asym", "rate", "burst_index"]].median().round(3))
var = {"main: t2p<0.4 or rate>10": hc2.classify(U.t2p, U.rate),
       "waveform only: t2p<0.4": np.where(U.t2p < 0.4, "int", "pyr"),
       "t2p<0.4 and rate>1, or rate>10": np.where(((U.t2p < 0.4) & (U.rate > 1)) | (U.rate > 10), "int", "pyr")}
for k, v in var.items():
    print(f"{k:35s} pyr {int((v == 'pyr').sum()):4d}  int {int((v == 'int').sum()):4d}  "
          f"int fraction {(v == 'int').mean():.3f}")
U["type_wave"] = var["waveform only: t2p<0.4"]
U["type_v3"] = var["t2p<0.4 and rate>1, or rate>10"]
print("paper, open field (Table S1, four rats): 1083 principal and 188 interneurons, int fraction",
      round(188 / (1083 + 188), 3))

## tables1-cell-counts

> The study recorded 2744 EC neurons ... and 3864 CA1, 876 CA3 and 175 DG neurons; CA1 held 3270
> principal cells and 464 interneurons, of which ec13 gave 940 and 205, ec14 832 and 80, ec16 528
> and 44, and i01 970 and 135; 1083 CA1 principal cells and 188 CA1 interneurons were tested in the
> open field.

hc-2 holds only CA1 open-field sessions of rats ec13, ec14 and ec16. The counts across tasks, the
EC, CA3 and DG counts and rat i01 are not in it. Counted session by session (O2 a) and once per
unit (b; sessions of one day share cluster ids, data descriptor 2c).

In [ ]:
U["uid"] = U.rat + "|" + U.day + "|" + U.shank.astype(str) + "|" + U.cluster.astype(str)
def counts(col):
    a = U.groupby(["rat", col]).size().unstack(fill_value=0)
    b = U.drop_duplicates("uid").groupby(["rat", col]).size().unstack(fill_value=0)
    return a, b
a, b = counts("type")
print("session by session (O2 a):"); print(a, "\n total", a.sum().to_dict())
print("\nonce per unit (O2 b):"); print(b, "\n total", b.sum().to_dict())
exp = {("ec13", "pyr"): 940, ("ec13", "int"): 205, ("ec14", "pyr"): 832, ("ec14", "int"): 80,
       ("ec16", "pyr"): 528, ("ec16", "int"): 44}
rows = []
for (rat, ty), v in exp.items():
    rows.append({"count": f"{rat} {ty} (all tasks)", "paper": v, "hc-2 session by session": a.loc[rat, ty],
                 "hc-2 once per unit": b.loc[rat, ty]})
rows.append({"count": "CA1 pyr, open field (paper: 4 rats incl. i01)", "paper": 1083,
             "hc-2 session by session": a["pyr"].sum(), "hc-2 once per unit": b["pyr"].sum()})
rows.append({"count": "CA1 int, open field (paper: 4 rats incl. i01)", "paper": 188,
             "hc-2 session by session": a["int"].sum(), "hc-2 once per unit": b["int"].sum()})
CNT = pd.DataFrame(rows)
print(); print(CNT.to_string())
for k in ["type_wave", "type_v3"]:
    print(f"\nclassification variant {k}: session by session", U.groupby(k).size().to_dict())

**Result: Not testable against the paper's numbers, so blocked (`data_missing`).** The per-rat counts of Table S1 cover all tasks and the open-field counts include rat i01, neither of which hc-2 holds. hc-2 gives 739 putative pyramidal cells and 181 putative interneurons counted session by session (545 and 127 once per unit) in three rats, against 1083 and 188 in the paper's open field with four rats.

## fig3d-hippocampal-modulation

> Among hippocampal principal cells, CA1 pyramidal cells have a lower mean resultant length than
> CA3 and DG cells and a lower percentage of theta-modulated cells than CA3 and DG.

hc-2 has no CA3 or DG cells, so the claim is tested on its two CA1 expected values (criterion):
mean R of significantly modulated CA1 pyramidal cells 0.19 +/- 0.019 and 84 +/- 8.4% modulated.
Inclusion as the Figure 3 legend: at least 50 spikes and rate > 0.1 Hz in theta epochs, Rayleigh
p < 0.01. Theta reference is the CA1 pyramidal layer LFP (no EC3 in hc-2).

In [ ]:
def modstats(df):
    inc = df[df.included]
    mod = inc[inc.modulated]
    return pd.Series({"included": len(inc), "modulated": len(mod),
                      "pct_modulated": 100 * len(mod) / max(len(inc), 1),
                      "mean_R": mod.R.mean(), "sd_R": mod.R.std()})
print("main (theta ratio > 2, speed > 3 cm/s; classification main):")
M3 = U.groupby("type").apply(modstats)
print(M3.round(3))
print("\nby rat, pyramidal cells:")
print(U[U.type == "pyr"].groupby("rat").apply(modstats).round(3))
print("\nonce per unit (first session of each unit):")
print(U.drop_duplicates("uid").groupby("type").apply(modstats).round(3))
for k in ["type_wave", "type_v3"]:
    print(f"\nclassification {k}, pyramidal cells:")
    print(modstats(U[U[k] == "pyr"]).round(3).to_dict())

In [ ]:
VAR = {}
for name, kw in {"ratio > 3, speed > 3": dict(ratio_thr=3.0), "ratio > 2, no speed": dict(speed_thr=None),
                 "ratio > 1.5, speed > 3": dict(ratio_thr=1.5), "ratio > 2, speed > 5": dict(speed_thr=5.0)}.items():
    VAR[name] = theta_all(**kw)
    print(f"theta epochs {name}: {sum(r['theta_s'] for r in VAR[name][0].values()) / 3600:.2f} h")
rows = []
for name, (th, uu) in {"ratio > 2, speed > 3 (main)": (TH, U), **VAR}.items():
    m = modstats(uu[uu.type == "pyr"])
    rows.append({"theta epochs": name, **m.to_dict()})
print(pd.DataFrame(rows).round(3).to_string())

**Result: Partially reproduced, resting on assumptions.** Mean R of the 427 significantly modulated CA1 pyramidal cells is 0.193 (expected 0.19 +/- 0.019, inside), and 93.0% of the 459 included cells are modulated (expected 84 +/- 8.4, just above the bound of 92.4). Under the theta-epoch variants the percentage is 91.1 to 95.8 and R 0.172 to 0.200; two of the four variants put both values in tolerance. The CA3 and DG comparisons cannot be tested.

## fig3-interneurons-min-ascending

> Interneurons in all regions, and principal cells other than EC3, are least active on the
> ascending phase of EC3 theta.

Population phase histogram of all theta spikes of each CA1 group (10 degree bins, all units), and
the bin of minimum probability. Readings for the ascending phase: (a) 180 to 360 degrees, (b) the
shaded column of Figure 3, about 265 to 360. hc-2 has no EC3 LFP; the reference is CA1 pyramidal
layer theta, as the criterion allows. CA1 theta leads EC3 theta by 0 to 80 degrees (Figure S10),
so a spike at CA1 phase x is at EC3 phase x minus 0 to 80 degrees; the minimum is also given in
EC3 coordinates over that range of shifts.

In [ ]:
BINS = np.arange(0, 361, 10)
CEN = BINS[:-1] + 5
def pooled(df, key="ph", th=None, sel=None):
    th = TH if th is None else th
    out = []
    for _, r in df.iterrows():
        u = th[r.session]["units"][r.i]
        ph = u[key]
        if sel is not None:
            ph = ph[sel(u)]
        out.append(ph)
    return np.concatenate(out) if out else np.array([])

def hist(ph):
    h, _ = np.histogram(ph[np.isfinite(ph)], BINS)
    return h / max(h.sum(), 1)

def in_range(x, lo, hi):
    return lo <= x % 360 <= hi

for ty in ["pyr", "int"]:
    ph = pooled(U[U.type == ty])
    h = hist(ph)
    mn = CEN[np.argmin(h)]; mx = CEN[np.argmax(h)]
    mu, R, p = hc2.circ_stats(ph)
    print(f"{ty}: {len(ph)} spikes from {int((U.type == ty).sum())} units; minimum bin {mn} deg (prob {h.min():.4f}), "
          f"maximum bin {mx} deg (prob {h.max():.4f}), circular mean {mu:.0f}, R {R:.3f}")
    print(f"   minimum within (a) 180-360: {in_range(mn, 180, 360)}; (b) 265-360: {in_range(mn, 265, 360)}")
    print(f"   minimum in EC3 coordinates if CA1 leads by 0 to 80 deg: {(mn - 80) % 360:.0f} to {mn % 360:.0f}")
    print("   histogram (10 deg bins from 0):", np.round(h * 100, 2).tolist())
    for rat in ["ec13", "ec14", "ec16"]:
        hr = hist(pooled(U[(U.type == ty) & (U.rat == rat)]))
        print(f"   {rat}: minimum bin {CEN[np.argmin(hr)]}")
for k in ["type_wave", "type_v3"]:
    for ty in ["pyr", "int"]:
        print(f"classification {k} {ty}: minimum bin {CEN[np.argmin(hist(pooled(U[U[k] == ty])))]}")
for name, (th, uu) in VAR.items():
    print(f"theta epochs {name}: pyr minimum {CEN[np.argmin(hist(pooled(uu[uu.type == 'pyr'], th=th)))]}, "
          f"int minimum {CEN[np.argmin(hist(pooled(uu[uu.type == 'int'], th=th)))]}")
fig, ax = plt.subplots(figsize=(6, 3))
for ty, c in [("pyr", "k"), ("int", "r")]:
    h = hist(pooled(U[U.type == ty]))
    ax.step(np.r_[CEN, CEN + 360], np.r_[h, h], where="mid", color=c, label=ty)
ax.axvspan(180, 360, color="0.9"); ax.set_xlabel("CA1 pyramidal layer theta phase (deg; peak 0)"); ax.legend()
ax.set_ylabel("probability"); plt.show()

**Result: Reproduced for the CA1 groups hc-2 holds.** The population minimum is at 335 degrees for pyramidal cells and 315 for interneurons of CA1 pyramidal layer theta, inside both readings of the ascending phase (180-360 and 265-360), in each rat (305 to 355) and under every classification and theta-epoch variant. In EC3 coordinates (CA1 leading by 0 to 80 degrees) the minima would be 255-335 and 235-315, inside reading (a) for any shift.

## figs9-phase-method

> Preferred theta phases are similar whether phase is computed from theta peaks, troughs or the
> Hilbert transform of 5-11 Hz filtered LFP.

For each CA1 group, the circular mean preferred phase of significantly modulated cells under each
of the three methods; reproduced when they agree within 90 degrees (reading b) for every group;
45 degrees (a) is reported.

In [ ]:
rows = []
for ty in ["pyr", "int"]:
    res = {}
    for key, name in [("ph", "maxima"), ("ph_min", "minima"), ("ph_hil", "Hilbert")]:
        prefs = []
        for _, r in U[(U.type == ty) & U.included].iterrows():
            mu, R, p = hc2.circ_stats(TH[r.session]["units"][r.i][key])
            if p < 0.01:
                prefs.append(mu)
        gm, gR, _ = hc2.circ_stats(np.array(prefs))
        res[name] = gm
        rows.append({"group": ty, "method": name, "modulated": len(prefs), "group_mean_phase": gm, "R_of_prefs": gR})
    d = [abs(hc2.circ_diff(res[a], res[b])) for a, b in [("maxima", "minima"), ("maxima", "Hilbert"), ("minima", "Hilbert")]]
    print(f"{ty}: largest difference between methods {max(d):.1f} deg; within 90: {max(d) < 90}; within 45: {max(d) < 45}")
print(pd.DataFrame(rows).round(2).to_string())

**Result: Reproduced for CA1.** The group mean preferred phases by the maxima, minima and Hilbert methods differ by at most 0.7 degrees for pyramidal cells and 0.2 for interneurons, within both 45 and 90 degrees.

## fig5-rate-phase-shift

> As instantaneous firing rate rises, EC2 and CA1 principal cells shift to progressively earlier
> theta phases (by more than 150 degrees), EC2 cells also losing phase locking, while DG and CA3
> cells shift less than 90 degrees without a change in modulation depth, and EC3 cells as a
> population change little.

CA1 pyramidal cells only. Spikes binned by instantaneous rate (1 to >= 10 spikes per 250 ms); per
category, cells with at least 50 spikes in it and Rayleigh p < 0.01; group circular mean of their
preferred phases and mean R. The shift from 4 Hz to >= 40 Hz is given as the circular difference
and as the path summed over the ten categories (which can exceed 180 degrees). Reproduced for
CA1 when it advances by more than 135 degrees.

In [ ]:
def rate_table(df, th=None):
    th = TH if th is None else th
    rows = []
    for k in range(1, 11):
        prefs, Rs = [], []
        for _, r in df.iterrows():
            u = th[r.session]["units"][r.i]
            ph = u["ph"][u["count"] == k]
            if len(ph) < 50:
                continue
            mu, R, p = hc2.circ_stats(ph)
            if p < 0.01:
                prefs.append(mu); Rs.append(R)
        gm = hc2.circ_stats(np.array(prefs))[0] if prefs else np.nan
        rows.append({"category": k, "Hz": 4 * k, "cells": len(prefs), "group_phase": gm,
                     "mean_R": np.mean(Rs) if Rs else np.nan})
    T = pd.DataFrame(rows)
    steps = hc2.circ_diff(T.group_phase.values[1:], T.group_phase.values[:-1])
    T["step_from_previous"] = np.r_[np.nan, steps]
    return T

def shift_summary(T):
    direct = hc2.circ_diff(T.group_phase.iloc[-1], T.group_phase.iloc[0])
    path = np.nansum(T.step_from_previous.values)
    return direct, path

RT = {}
for ty in ["pyr", "int"]:
    RT[ty] = rate_table(U[(U.type == ty)])
    print(ty); print(RT[ty].round(2).to_string())
    d, pth = shift_summary(RT[ty])
    print(f"  shift 4 Hz -> >=40 Hz: circular {d:.1f} deg, summed path {pth:.1f} deg (negative = advance to earlier phase)")
    print(f"  Spearman of mean R with category: rho {stats.spearmanr(RT[ty].Hz, RT[ty].mean_R)[0]:.2f}")
# single cells modulated in both categories (Figure S12A variant)
sc = []
for _, r in U[(U.type == "pyr")].iterrows():
    u = TH[r.session]["units"][r.i]
    a, b = u["ph"][u["count"] == 1], u["ph"][u["count"] == 10]
    if len(a) >= 50 and len(b) >= 50:
        ma, _, pa = hc2.circ_stats(a); mb, _, pb = hc2.circ_stats(b)
        if pa < 0.01 and pb < 0.01:
            sc.append(hc2.circ_diff(mb, ma))
sc = np.array(sc)
print(f"\nsingle CA1 pyramidal cells modulated at 4 and >=40 Hz: n = {len(sc)}, circular mean shift "
      f"{hc2.circ_stats(sc % 360)[0] - 360 if hc2.circ_stats(sc % 360)[0] > 180 else hc2.circ_stats(sc % 360)[0]:.1f} deg, "
      f"median {np.median(sc):.1f}, fraction advancing {np.mean(sc < 0):.2f}")
for rat in ["ec13", "ec14", "ec16"]:
    T = rate_table(U[(U.type == "pyr") & (U.rat == rat)])
    print(f"{rat}: shift circular {shift_summary(T)[0]:.1f}, path {shift_summary(T)[1]:.1f} "
          f"(cells at 4 Hz {T.cells.iloc[0]}, at >=40 Hz {T.cells.iloc[-1]})")
for k in ["type_wave", "type_v3"]:
    print(f"classification {k}: pyr shift circular/path {np.round(shift_summary(rate_table(U[U[k] == 'pyr'])), 1)}")
for name, (th, uu) in VAR.items():
    print(f"theta epochs {name}: pyr shift circular/path {np.round(shift_summary(rate_table(uu[uu.type == 'pyr'], th)), 1)}")

**Result: Partially reproduced.** CA1 pyramidal cells move to earlier phases as instantaneous rate rises, every step between rate categories being an advance, by 130.2 degrees from 4 Hz to >= 40 Hz, below the criterion's 135 (the paper: more than 150). The 139 single cells modulated in both categories advance by a median of 134.6 degrees (91% advance); the theta-epoch variants give 129.9 to 131.1 and the rats 125.6, 147.3 and 129.6. EC2, CA3, DG and EC3 cannot be tested.

## fig5b-burst-double-peak

> Burst spikes (ISI < 6 ms) tend to occupy the same theta phase as fast spikes (>= 40 Hz), but in
> EC2 and CA1 principal cells the burst probability has two small peaks.

Pooled phase histograms (10 degree bins, all CA1 pyramidal cells) of burst spikes and of >= 40 Hz
spikes. Similar phase: circular means within 90 degrees. Two peaks: (a) two local maxima of the
3-bin smoothed histogram at least 60 degrees apart, each above the histogram mean; (b) a mixture
of two von Mises components preferred to one by BIC.

In [ ]:
def local_peaks(h, min_sep=60):
    hs = (np.roll(h, 1) + h + np.roll(h, -1)) / 3
    pk = [i for i in range(len(hs)) if hs[i] > hs[i - 1] and hs[i] >= hs[(i + 1) % len(hs)] and hs[i] > hs.mean()]
    pk = sorted(pk, key=lambda i: -hs[i])
    sel = []
    for i in pk:
        if all(min(abs(CEN[i] - CEN[j]), 360 - abs(CEN[i] - CEN[j])) >= min_sep for j in sel):
            sel.append(i)
    return [int(CEN[i]) for i in sel], hs

def vm_mixture_bic(ph_deg, rng, max_n=20000):
    x = np.radians(ph_deg[np.isfinite(ph_deg)])
    if len(x) > max_n:
        x = rng.choice(x, max_n, replace=False)
    n = len(x)
    from scipy.special import i0e
    def ll_k(K, iters=200):
        w = np.ones(K) / K; mu = np.linspace(0, 2 * np.pi, K, endpoint=False) + rng.uniform(0, 2 * np.pi); kap = np.ones(K) * 2
        for _ in range(iters):
            dens = np.array([w[k] * np.exp(kap[k] * (np.cos(x - mu[k]) - 1)) / (2 * np.pi * i0e(kap[k])) for k in range(K)])
            tot = dens.sum(0) + 1e-300
            g = dens / tot
            for k in range(K):
                w[k] = g[k].mean()
                C, Sn = (g[k] * np.cos(x)).sum(), (g[k] * np.sin(x)).sum()
                mu[k] = np.arctan2(Sn, C)
                Rb = np.hypot(C, Sn) / g[k].sum()
                kap[k] = Rb * (2 - Rb ** 2) / (1 - Rb ** 2 + 1e-9)
        return np.log(tot).sum(), np.degrees(mu) % 360, w
    l1, m1, _ = ll_k(1); l2, m2, w2 = ll_k(2)
    bic1 = -2 * l1 + 2 * np.log(n); bic2 = -2 * l2 + 5 * np.log(n)
    return bic1, bic2, m2, w2

rng = np.random.default_rng(55)
pyr = U[U.type == "pyr"]
ph_b = pooled(pyr, sel=lambda u: u["burst"])
ph_f = pooled(pyr, sel=lambda u: u["count"] == 10)
mb = hc2.circ_stats(ph_b)[0]; mf = hc2.circ_stats(ph_f)[0]
print(f"CA1 pyr burst spikes {len(ph_b)}, circular mean {mb:.0f}; >=40 Hz spikes {len(ph_f)}, circular mean {mf:.0f}; "
      f"difference {abs(hc2.circ_diff(mb, mf)):.0f} deg (within 90: {abs(hc2.circ_diff(mb, mf)) < 90})")
hb = hist(ph_b)
pk, hs = local_peaks(hb)
print("burst histogram (%):", np.round(hb * 100, 2).tolist())
print(f"(a) local maxima of the smoothed histogram above its mean, >= 60 deg apart: {pk}; two peaks: {len(pk) >= 2}")
b1, b2, m2, w2 = vm_mixture_bic(ph_b, rng)
print(f"(b) BIC one component {b1:.0f}, two components {b2:.0f}; two preferred: {b2 < b1}; "
      f"component means {np.round(m2)} weights {np.round(w2, 2)}")
for rat in ["ec13", "ec14", "ec16"]:
    pr = pooled(pyr[pyr.rat == rat], sel=lambda u: u["burst"])
    print(f"{rat}: burst spikes {len(pr)}, peaks (a) {local_peaks(hist(pr))[0]}")
for k in ["type_wave", "type_v3"]:
    pr = pooled(U[U[k] == "pyr"], sel=lambda u: u["burst"])
    print(f"classification {k}: peaks (a) {local_peaks(hist(pr))[0]}")

**Result: Partially reproduced.** The burst-spike histogram of CA1 pyramidal cells has two peaks (235 and 25 degrees; two von Mises components preferred by BIC), but burst spikes do not share the phase of >= 40 Hz spikes: their circular means are 154 degrees apart (criterion within 90). EC2 cannot be tested.

## figs13-interneuron-rate

> Firing rate and bursting shift interneurons' theta phase only marginally (less than 30 degrees).

CA1 interneurons: group mean preferred phases (cells with >= 50 spikes and Rayleigh p < 0.01 in
the category) at 4 Hz, >= 40 Hz and for burst spikes; reproduced when they differ pairwise by
less than 33 degrees.

In [ ]:
def group_phase(df, sel, th=None):
    th = TH if th is None else th
    prefs = []
    for _, r in df.iterrows():
        u = th[r.session]["units"][r.i]
        ph = u["ph"][sel(u)]
        if len(ph) >= 50:
            mu, R, p = hc2.circ_stats(ph)
            if p < 0.01:
                prefs.append(mu)
    return (hc2.circ_stats(np.array(prefs))[0] if prefs else np.nan), len(prefs)

def interneuron_shifts(df, th=None):
    g = {"4 Hz": group_phase(df, lambda u: u["count"] == 1, th), ">=40 Hz": group_phase(df, lambda u: u["count"] == 10, th),
         "burst": group_phase(df, lambda u: u["burst"], th)}
    pairs = {f"{a} vs {b}": abs(hc2.circ_diff(g[a][0], g[b][0])) for a, b in [("4 Hz", ">=40 Hz"), ("4 Hz", "burst"), (">=40 Hz", "burst")]}
    return g, pairs

g, pairs = interneuron_shifts(U[U.type == "int"])
print("CA1 interneurons, group mean phase (deg) and cells:", {k: (round(v[0], 1), v[1]) for k, v in g.items()})
print("pairwise differences (deg):", {k: round(v, 1) for k, v in pairs.items()}, " largest", round(max(pairs.values()), 1))
print("for comparison, CA1 pyramidal cells:", {k: round(v, 1) for k, v in interneuron_shifts(U[U.type == "pyr"])[1].items()})
for k in ["type_wave", "type_v3"]:
    print(f"classification {k}:", {kk: round(v, 1) for kk, v in interneuron_shifts(U[U[k] == "int"])[1].items()})
for name, (th, uu) in VAR.items():
    print(f"theta epochs {name}:", {kk: round(v, 1) for kk, v in interneuron_shifts(uu[uu.type == "int"], th)[1].items()})
for rat in ["ec13", "ec14", "ec16"]:
    gg, pp = interneuron_shifts(U[(U.type == "int") & (U.rat == rat)])
    print(f"{rat}: cells {[v[1] for v in gg.values()]}, differences", {k: round(v, 1) for k, v in pp.items()})

**Result: Not reproduced, resting on the assumed cell classification.** The group mean phase of CA1 interneurons moves 59.6 degrees between 4 Hz and >= 40 Hz and 46.5 between 4 Hz and burst spikes, above the 33 degree bound; the theta-epoch variants give 51.6 to 58.7, and only rat ec13 alone stays near the bound (27.2 and 34.0).

## figs12b-burst-single-phase

> Burst spikes (ISI < 6 ms) and single spikes (ISI > 20 ms) have different theta phase
> preferences in a significant number of CA1 and EC2 principal neurons.

CA1 pyramidal cells significantly modulated in both spike groups (>= 50 spikes each, Rayleigh
p < 0.01). Per cell the phases differ when (a) the circular difference exceeds 90 degrees, (b) a
Watson-Williams test gives p < 0.05. "A significant number": binomial test of the count against
5%, p < 0.05.

In [ ]:
def watson_williams(a_deg, b_deg):
    """Two-sample Watson-Williams test (Zar 1999, eq. 27.15, with the kappa correction)."""
    a, b = np.radians(a_deg), np.radians(b_deg)
    n1, n2 = len(a), len(b); N = n1 + n2
    R1 = abs(np.exp(1j * a).sum()); R2 = abs(np.exp(1j * b).sum())
    R = abs(np.exp(1j * np.r_[a, b]).sum())
    rw = (R1 + R2) / N
    # kappa from the weighted mean resultant length (Fisher 1993 approximation)
    if rw < 0.53:
        k = 2 * rw + rw ** 3 + 5 * rw ** 5 / 6
    elif rw < 0.85:
        k = -0.4 + 1.39 * rw + 0.43 / (1 - rw)
    else:
        k = 1 / (rw ** 3 - 4 * rw ** 2 + 3 * rw)
    K = 1 + 3 / (8 * k)
    F = K * (N - 2) * (R1 + R2 - R) / (N - R1 - R2)
    return F, stats.f.sf(F, 1, N - 2)

rows = []
for _, r in U[U.type == "pyr"].iterrows():
    u = TH[r.session]["units"][r.i]
    a, b = u["ph"][u["burst"]], u["ph"][u["single"]]
    if len(a) < 50 or len(b) < 50:
        continue
    ma, Ra, pa = hc2.circ_stats(a); mb, Rb, pb = hc2.circ_stats(b)
    if pa < 0.01 and pb < 0.01:
        rows.append({"rat": r.rat, "diff": hc2.circ_diff(ma, mb), "ww_p": watson_williams(a, b)[1]})
B = pd.DataFrame(rows)
n = len(B)
for name, k in [("(a) |difference| > 90 deg", int((B["diff"].abs() > 90).sum())), ("(b) Watson-Williams p < 0.05", int((B.ww_p < 0.05).sum()))]:
    p = stats.binomtest(k, n, 0.05, alternative="greater").pvalue
    print(f"{name}: {k} of {n} cells ({100 * k / n:.1f}%), binomial p against 5% = {p:.2g}")
print(f"burst minus single phase: median {B['diff'].median():.1f} deg, IQR {B['diff'].quantile(.25):.1f} to {B['diff'].quantile(.75):.1f}")
print(B.groupby("rat").apply(lambda x: pd.Series({"cells": len(x), "a": int((x['diff'].abs() > 90).sum()), "b": int((x.ww_p < 0.05).sum())})))

**Result: Reproduced for CA1.** Of 226 pyramidal cells modulated in both spike groups, 64 (28.3%) have burst and single-spike phases more than 90 degrees apart and 177 (78.3%) differ by Watson-Williams test, both far above 5% (binomial p 2.8e-30 and 5.7e-182). EC2 is not in hc-2.

## figs19-fast-firing-minority

> In each region and layer most principal neurons are silent or fire at low rates, with only a
> minority of neurons discharging at high frequency at a given time; in the EC, spikes at high
> firing rates are a minority.

Fraction of CA1 pyramidal theta spikes in each instantaneous rate category (all spikes of all
cells counted equally, Figure S19 legend). Readings: (a) high >= 40 Hz, low 4 to 12 Hz; (b) high
>= 20 Hz, low below 20 Hz. Reproduced when more spikes fall at low than at high rates. The EC
part is not testable. The neuron-level part: the proportion of included pyramidal cells
significantly modulated in each category.

In [ ]:
cnt = pooled(U[U.type == "pyr"], key="count")
frac = np.array([(cnt == k).mean() for k in range(1, 11)])
print("fraction of CA1 pyramidal theta spikes per category (4..>=40 Hz):", np.round(frac, 3).tolist(), " spikes", len(cnt))
a_low, a_high = frac[:3].sum(), frac[9]
b_low, b_high = frac[:4].sum(), frac[4:].sum()
print(f"(a) low 4-12 Hz {a_low:.3f}, high >=40 Hz {a_high:.3f}: low > high {a_low > a_high}")
print(f"(b) low <20 Hz {b_low:.3f}, high >=20 Hz {b_high:.3f}: low > high {b_low > b_high}")
for rat in ["ec13", "ec14", "ec16"]:
    c = pooled(U[(U.type == "pyr") & (U.rat == rat)], key="count")
    print(f"{rat}: (a) low {np.isin(c, [1, 2, 3]).mean():.3f} high {(c == 10).mean():.3f}; (b) low {(c <= 4).mean():.3f} high {(c >= 5).mean():.3f}")
inc = U[(U.type == "pyr") & U.included]
print(f"\nneuron level: proportion of {len(inc)} included pyramidal cells significantly modulated in each category:",
      np.round(RT["pyr"].cells.values / len(inc), 3).tolist())

**Result: Reproduced for CA1.** Of CA1 pyramidal theta spikes, 0.546 fall at 4 to 12 Hz and 0.049 at >= 40 Hz (a); 0.662 below 20 Hz and 0.338 at or above (b); the same holds in each rat. The EC part cannot be tested.

## figs7-individual-rats

> The theta phase preferences of the neuron groups are consistent across individual rats.

Significantly modulated CA1 pyramidal cells and interneurons per rat; circular mean preferred
phase; reproduced when every pair of rats agrees within 90 degrees (b); 45 degrees (a) reported.
The paper references EC3 theta except in i01; here CA1 pyramidal layer theta in every rat.

In [ ]:
for ty in ["pyr", "int"]:
    gm = {}
    for rat in ["ec13", "ec14", "ec16"]:
        m = U[(U.type == ty) & U.modulated & (U.rat == rat)]
        gm[rat] = hc2.circ_stats(m.phase.values)
        print(f"{ty} {rat}: {len(m)} modulated cells, group mean {gm[rat][0]:.1f} deg, R of preferred phases {gm[rat][1]:.2f}")
    d = {f"{a}-{b}": abs(hc2.circ_diff(gm[a][0], gm[b][0])) for a, b in [("ec13", "ec14"), ("ec13", "ec16"), ("ec14", "ec16")]}
    print(f"  pairwise differences {np.round(list(d.values()), 1).tolist()}; largest {max(d.values()):.1f}; "
          f"within 90: {max(d.values()) < 90}; within 45: {max(d.values()) < 45}")
for k in ["type_wave", "type_v3"]:
    for ty in ["pyr", "int"]:
        gm = [hc2.circ_stats(U[(U[k] == ty) & U.modulated & (U.rat == rat)].phase.values)[0] for rat in ["ec13", "ec14", "ec16"]]
        print(f"classification {k} {ty}: group means {np.round(gm, 1).tolist()}")
print("\nper session, pyramidal cells:")
print(U[(U.type == "pyr") & U.modulated].groupby(["rat", "session"]).phase.agg(
    n="size", mean=lambda x: hc2.circ_stats(x.values)[0]).round(1).to_string())

**Result: Reproduced for CA1.** The group mean preferred phases of pyramidal cells in the three rats differ by at most 8.5 degrees and those of interneurons by at most 31.9, within both 45 and 90 degrees.

## figs11-phase-time-agreement

> The time offsets of individual neuron pairs correlate closely with the differences between
> their preferred theta phases.

Pairs of simultaneously recorded CA1 units both significantly theta modulated whose theta-epoch
cross-correlogram has more than 100 events in [-150, 150] ms (Figure S11 legend). CCG in 1 ms bins
over [-200, 200] ms, normalised by (1 - |t|/T), smoothed with a Gaussian of SD 10 ms; the time
offset is its peak in [-150, 150] ms. The phase difference (target minus reference) is converted
to time with 1 cycle = 125 ms. "Close correlation": Pearson r > 0.5 with p < 0.05 (threshold of
the criterion).

In [ ]:
from scipy.ndimage import gaussian_filter1d
LAGS = np.arange(-200, 201)
def ccg(a, b, T):
    """Counts of b spikes at lags (ms) from each a spike, 1 ms bins over [-200, 200]."""
    lo = np.searchsorted(b, a - 0.2005); hi = np.searchsorted(b, a + 0.2005)
    tot = hi - lo
    if tot.sum() == 0:
        return np.zeros(len(LAGS))
    idx = np.repeat(np.arange(len(a)), tot)
    off = np.arange(tot.sum()) - np.repeat(np.cumsum(tot) - tot, tot)
    d = (b[lo[idx] + off] - a[idx]) * 1000
    h = np.histogram(d, np.arange(-200.5, 201.5))[0].astype(float)
    return h / (1 - np.abs(LAGS) / 1000 / T)

rows = []
win = np.abs(LAGS) <= 150
for s in TH:
    m = U[(U.session == s) & U.modulated]
    T = TH[s]["theta_s"]
    recs = list(m.itertuples())
    for x in range(len(recs)):
        for y in range(x + 1, len(recs)):
            ra, rb = recs[x], recs[y]
            a, b = TH[s]["units"][ra.i]["t"], TH[s]["units"][rb.i]["t"]
            h = ccg(a, b, T)
            if h[win].sum() <= 100:
                continue
            hs = gaussian_filter1d(h, 10, mode="nearest")
            lag = LAGS[win][np.argmax(hs[win])]
            pred = hc2.circ_diff(rb.phase, ra.phase) / 360 * 125
            rows.append({"session": s, "rat": ra.rat, "pair": f"{ra.type}-{rb.type}", "lag_ms": lag, "phase_ms": pred})
P = pd.DataFrame(rows)
P["pair"] = P.pair.replace({"int-pyr": "pyr-int"})
for grp, g in [("all", P)] + list(P.groupby("pair")):
    r, p = stats.pearsonr(g.phase_ms, g.lag_ms)
    rs = stats.spearmanr(g.phase_ms, g.lag_ms)[0]
    inside = (g.lag_ms.abs() <= 62.5).mean()
    print(f"{grp:8s} pairs {len(g):5d}: Pearson r {r:.3f} (p {p:.2g}), Spearman {rs:.3f}; "
          f"lags within +/-62.5 ms {inside:.2f}")
# The criterion's readings: the CCG lag unwrapped to one theta cycle, [-62.5, 62.5) ms, then Pearson;
# and a circular correlation (Fisher and Lee) of the two as angles of a 125 ms cycle.
P["lag_wrapped"] = (P.lag_ms + 62.5) % 125 - 62.5
def circ_corr(a, b):
    a, b = np.radians(a), np.radians(b)
    ma, mb = np.angle(np.exp(1j * a).mean()), np.angle(np.exp(1j * b).mean())
    sa, sb = np.sin(a - ma), np.sin(b - mb)
    return (sa * sb).sum() / np.sqrt((sa ** 2).sum() * (sb ** 2).sum())
print("\nlag unwrapped to one cycle (criterion reading):")
for grp, g in [("all", P)] + list(P.groupby("pair")):
    r, p = stats.pearsonr(g.phase_ms, g.lag_wrapped)
    rc = circ_corr(g.phase_ms / 125 * 360, g.lag_wrapped / 125 * 360)
    print(f"{grp:8s} pairs {len(g):5d}: Pearson r {r:.3f} (p {p:.2g}); circular correlation {rc:.3f}; "
          f"median |lag - phase time| {np.median(np.abs(g.lag_wrapped - g.phase_ms)):.1f} ms")
for rat, g in P.groupby("rat"):
    print(f"{rat}: pairs {len(g)}, Pearson r (unwrapped lag) {stats.pearsonr(g.phase_ms, g.lag_wrapped)[0]:.3f}")
g2 = P[P.lag_ms.abs() <= 62.5]
r, p = stats.pearsonr(g2.phase_ms, g2.lag_ms)
print(f"variant, only pairs whose peak lies within +/-62.5 ms: {len(g2)}, Pearson r {r:.3f} (p {p:.2g})")
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(P.phase_ms, P.lag_wrapped, ".", ms=2); ax.set_xlabel("phase difference as time (ms)"); ax.set_ylabel("CCG peak lag, one cycle (ms)")
print("plotted ranges: phase", P.phase_ms.min().round(1), P.phase_ms.max().round(1), "lag", P.lag_wrapped.min(), P.lag_wrapped.max())
plt.show()

**Result: Partially reproduced.** With the CCG lag unwrapped to one theta cycle (the criterion's reading), the phase-derived and CCG time offsets correlate at r = 0.614 for 1840 pyramidal cell-interneuron pairs, but only r = 0.146 for 3406 pyramidal cell pairs and 0.361 over all pairs (criterion r > 0.5). Without unwrapping r is 0.061 over all pairs.

## figs2-ripple-sequence

> During sharp wave ripples, onset and peak delays of principal cells and interneurons increase by
> 15 to 25 ms along the CA1-EC5-EC3-EC2 axis; CA1 pyramidal cells increase firing about 6-fold and
> CA1 interneurons about 3-fold, while EC principal cells are recruited more weakly than EC
> interneurons in every layer.

Ripples in the CA1 pyramidal layer LFP in non-theta periods (theta ratio not above threshold),
140-230 Hz, RMS in 17 ms windows, 7 SD above the mean. Spikes aligned to the ripple power peak in
5 ms bins over +/-500 ms; mean rate per cell, averaged over cells of each type. Fold increase =
peak of the group mean within +/-50 ms over the baseline: (a) -500 to -250 ms, (b) each cell's
mean rate outside +/-100 ms of ripples, averaged over cells. Only the CA1 fold increases are
testable (expected 6 +/- 0.6 and 3 +/- 0.3).

In [ ]:
EDGES = np.arange(-0.5, 0.5001, 0.005)
CT = (EDGES[:-1] + EDGES[1:]) / 2
def peri(t, ev):
    lo = np.searchsorted(t, ev - 0.5); hi = np.searchsorted(t, ev + 0.5)
    d = np.concatenate([t[a:b] - e for a, b, e in zip(lo, hi, ev)]) if len(ev) else np.array([])
    return np.histogram(d, EDGES)[0] / max(len(ev), 1) / 0.005

PR, BASE, NRIP = {"pyr": [], "int": []}, {"pyr": [], "int": []}, {}
for s, d in RAW.items():
    t_, ratio = hc2.theta_ratio(d["lfp"])
    nt = np.interp(np.arange(len(d["lfp"])) / hc2.FS_LFP, t_, ratio) <= hc2.THETA_RATIO
    ev = hc2.detect_ripples(d["lfp"], nt)
    NRIP[s] = (len(ev), nt.mean() * len(d["lfp"]) / hc2.FS_LFP)
    dur = len(d["lfp"]) / hc2.FS_LFP
    near = np.c_[ev - 0.1, ev + 0.1]
    out_s = dur - min(len(ev) * 0.2, dur)
    for _, r in U[U.session == s].iterrows():
        tt = d["units"][r.i]["spikes"]
        PR[r.type].append(peri(tt, ev))
        BASE[r.type].append((~hc2.in_epochs(tt, near)).sum() / out_s if len(ev) else np.nan)
print("ripples per session (count, non-theta seconds):", {s: (n, round(x)) for s, (n, x) in NRIP.items()})
print(f"total ripples {sum(n for n, _ in NRIP.values())}")
FOLD = {}
for ty in ["pyr", "int"]:
    M = np.array(PR[ty]); g = M.mean(0)
    pk = g[np.abs(CT) <= 0.05].max(); tpk = CT[np.abs(CT) <= 0.05][np.argmax(g[np.abs(CT) <= 0.05])]
    ba = g[(CT >= -0.5) & (CT <= -0.25)].mean(); bb = np.nanmean(BASE[ty])
    FOLD[ty] = (pk / ba, pk / bb)
    print(f"CA1 {ty}: {len(M)} cells; peak {pk:.2f} Hz at {tpk * 1000:.0f} ms; baseline (a) {ba:.2f} Hz, (b) {bb:.2f} Hz; "
          f"fold (a) {pk / ba:.2f}, (b) {pk / bb:.2f}")
    sem = M.std(0) / np.sqrt(len(M))
    print(f"   group mean rate range {g.min():.2f} to {g.max():.2f} Hz; SEM at peak {sem[np.argmax(g)]:.2f}")
for rat in ["ec13", "ec14", "ec16"]:
    for ty in ["pyr", "int"]:
        idx = [k for k, (_, r) in enumerate(U[U.type == ty].iterrows()) if r.rat == rat]
        g = np.array(PR[ty])[idx].mean(0)
        print(f"{rat} {ty}: fold (a) {g[np.abs(CT) <= 0.05].max() / g[(CT >= -0.5) & (CT <= -0.25)].mean():.2f}")
# The bin of Figure S2 is not given: the fold increase with 10 and 20 ms bins (sums of 5 ms bins).
for w in [2, 4]:
    for ty in ["pyr", "int"]:
        g = np.array(PR[ty]).mean(0); n = len(g) // w * w
        gb = g[:n].reshape(-1, w).mean(1); cb = CT[:n].reshape(-1, w).mean(1)
        pk = gb[np.abs(cb) <= 0.05 + 1e-9].max(); ba = gb[(cb >= -0.5) & (cb <= -0.25)].mean()
        print(f"{5 * w} ms bins, CA1 {ty}: fold (a) {pk / ba:.2f}, (b) {pk / np.nanmean(BASE[ty]):.2f}")
fig, ax = plt.subplots(figsize=(5, 3))
for ty, c in [("pyr", "k"), ("int", "r")]:
    ax.plot(CT * 1000, np.array(PR[ty]).mean(0), c, label=ty)
ax.set_xlabel("time from ripple power peak (ms)"); ax.set_ylabel("mean rate (Hz)"); ax.legend(); plt.show()

**Result: Not reproduced.** Around 2067 ripples, CA1 pyramidal cells fire 10.05 times (baseline -500 to -250 ms) or 7.41 times (rate outside ripples) their baseline, and interneurons 4.41 or 2.12 times, against 6 +/- 0.6 and 3 +/- 0.3; no baseline reading or bin width (5, 10, 20 ms) puts both within tolerance. Pyramidal cells increase more than interneurons, as the paper states. The EC delays and recruitment cannot be tested.

## Summary

| Claim | Outcome |
|---|---|
| tables1-cell-counts | blocked (data missing): counts need other tasks and rat i01 |
| fig3d-hippocampal-modulation | partially reproduced (assumed) |
| fig3-interneurons-min-ascending | reproduced on CA1 (assumed) |
| figs9-phase-method | reproduced on CA1 (assumed) |
| fig5-rate-phase-shift | partially reproduced (assumed; CA1 advance 130 degrees) |
| fig5b-burst-double-peak | partially reproduced (assumed) |
| figs13-interneuron-rate | not reproduced (assumed) |
| figs12b-burst-single-phase | reproduced on CA1 (assumed) |
| figs19-fast-firing-minority | reproduced on CA1 (assumed) |
| figs7-individual-rats | reproduced on CA1 (assumed) |
| figs11-phase-time-agreement | partially reproduced (assumed) |
| figs2-ripple-sequence | not reproduced (assumed) |
| 25 claims on EC, CA3, DG, other tasks and outside CSD data | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Theta phase is referred to CA1 pyramidal layer
theta, not EC3 theta, which hc-2 lacks. Theta epochs come from a fixed threshold on the theta
ratio and a running speed threshold, without the paper's manual adjustment. CA1 cell types come
from a waveform and rate rule, as the paper's labels are not in hc-2. Ripples are detected with
the paper's band, window and threshold; the baseline for the fold increase and the bin are not
given and are varied.

**What was found about the data set.** hc-2 holds only the CA1 open-field part of the paper's
data (three of five rats, one of four tasks, one of four regions). It has no cell-type or layer
labels (the .xml unit fields are empty), does not say which channel is in the pyramidal layer,
and gives positions in pixels without a scale. Sessions ec015.041 and ec015.047 carry the prefix
of another animal but are listed under rat ec14. Units of sessions recorded on one day share
cluster ids, so the 920 session units are 672 distinct units. The LFP, positions and spikes share
one clock in every session by three contrasts.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")